# PostgreSQL Image Insert Benchmark

Equivalent of `insert_test_3.py` for PostgreSQL.
Inserts 1 M synthetic JPEG images (64×64) into an **UNLOGGED** table using
multi-threaded `psycopg2` batch inserts.

Reads connection details from environment variables `DB_HOST`, `DB_NAME`,
`DB_USER`, `DB_PASSWORD`; falls back to the docker-compose defaults.

In [17]:
import io
import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
from queue import Queue

import numpy as np
import psycopg2
import psycopg2.extras
from PIL import Image
from tqdm import tqdm

DB_HOST     = os.environ.get('DB_HOST',     'prototyping-pg-1')
DB_NAME     = os.environ.get('DB_NAME',     'testdb')
DB_USER     = os.environ.get('DB_USER',     'testuser')
DB_PASSWORD = os.environ.get('DB_PASSWORD', 'mypassword')
DSN = f'dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} host={DB_HOST}'

TOTAL_IMAGES = 1_000_000
IMAGE_SIZE   = (64, 64, 3)
N_WORKERS    = cpu_count()
BATCH_SIZE   = 1024
TABLE_NAME = 'bench_images64'

print(f'Workers: {N_WORKERS}, Batch size: {BATCH_SIZE}')


Workers: 32, Batch size: 1024


## Create Table

Creates an `UNLOGGED` table `bench_images` (skipping WAL for maximum insert throughput).
Run the drop cell first if you want to reset.

In [18]:
# Optional: drop first to start fresh
with psycopg2.connect(DSN) as conn:
    with conn.cursor() as cur:
        cur.execute(f'DROP TABLE IF EXISTS {TABLE_NAME};')
    conn.commit()
print(f'Dropped {TABLE_NAME}.')

Dropped bench_images64.


In [19]:
TABLE_DDL = f"""
CREATE UNLOGGED TABLE IF NOT EXISTS {TABLE_NAME} (
    id          BIGINT PRIMARY KEY,
    image_data  BYTEA NOT NULL
);
"""

with psycopg2.connect(DSN) as conn:
    with conn.cursor() as cur:
        cur.execute(TABLE_DDL)
    conn.commit()

print(f'Table {TABLE_NAME} ready.')

Table bench_images64 ready.


## Write Benchmark

In [20]:
def generate_jpeg_bytes():
    arr = np.random.randint(0, 256, IMAGE_SIZE, dtype=np.uint8)
    img = Image.fromarray(arr)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()


def insert_worker(start_idx, n_images, progress_queue):
    conn = psycopg2.connect(DSN)
    cur  = conn.cursor()
    # reuse a single pre-generated image (mirrors insert_test_3.py behaviour)
    jpeg_bytes = generate_jpeg_bytes()

    i = 0
    while i < n_images:
        end   = min(i + BATCH_SIZE, n_images)
        batch = [
            (start_idx + j, psycopg2.Binary(jpeg_bytes))
            for j in range(i, end)
        ]
        psycopg2.extras.execute_values(
            cur,
            f'INSERT INTO {TABLE_NAME} (id, image_data) VALUES %s ON CONFLICT DO NOTHING',
            batch,
        )
        conn.commit()
        progress_queue.put(len(batch))
        i = end

    cur.close()
    conn.close()


progress_queue   = Queue()
images_per_worker = TOTAL_IMAGES // N_WORKERS
pbar = tqdm(total=TOTAL_IMAGES, desc='Inserting')


def progress_updater():
    while True:
        n = progress_queue.get()
        if n is None:
            break
        pbar.update(n)


updater = threading.Thread(target=progress_updater, daemon=True)
updater.start()

start_time = time.perf_counter()

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = [
        executor.submit(insert_worker, i * images_per_worker, images_per_worker, progress_queue)
        for i in range(N_WORKERS)
    ]
    for f in as_completed(futures):
        f.result()  # propagate exceptions

progress_queue.put(None)
updater.join()
pbar.close()

elapsed = time.perf_counter() - start_time
print(f'\nInserted {TOTAL_IMAGES:,} images in {elapsed:.2f} seconds')
print(f'Throughput: {TOTAL_IMAGES / elapsed:,.0f} images/sec')

Inserting: 100%|██████████| 1000000/1000000 [00:16<00:00, 60104.37it/s]


Inserted 1,000,000 images in 16.64 seconds
Throughput: 60,103 images/sec
